# Final Walk-Forward Retraining Handoff V1

This notebook consolidates the current strict retention gate, SOXLBacktest Qlib reference evidence, checkpoint selectors, state-conditioned objective screens, and reward-readiness screens into one retraining handoff.

Experiment governance:

- **Baselines**: canonical reference `soxl_backtest_policy_reimplementation_v1`; direct comparison against strict retention, state-conditioned multi-objective, checkpoint drift selectors, and short-volshock reward recipe screens.
- **Role**: diagnose and hand off the next training screen. This notebook does not promote, retain, or train an actor.
- **Target windows**: `wf_2020`, `wf_2022`, `wf_2023_2024`, plus strict YTD/common-window return alignment.
- **Changed layer**: `model_selection`, `reward`, and training-objective handoff only.
- **External reference**: qlib/SOXLBacktest buy-sell reference, drawdown-drift reward recipe, official replay gate evidence, checkpoint drift selector evidence.
- **Acceptance criteria for any future retained actor**: all required WF windows present, worst WF MaxDD `< 20%`, return aligned to current best requirement, and strict candidate filter actor-retention count above zero.
- **Failure mode**: continuing longer PPO from current checkpoint screens or retaining a SOXLBacktest reference as if it were a runnable actor.
- **Closest historical success**: fold-local checkpoint selectors pass some individual folds (`wf_2022` / `wf_2023_2024`) under drift gates.
- **Closest historical failure**: state-conditioned multi-objective and checkpoint drift candidates hit return targets but fail drawdown/drift constraints.
- **Duplicate check**: this notebook does not rename an existing profile or retune a threshold; it aggregates cross-card no-promotion evidence into a single next-training contract.


In [1]:

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook execution directory")

ROOT = find_repo_root(Path.cwd())
CARD = ROOT / "runtime" / "model_selection" / "card" / "final_walkforward_retraining_handoff_v1"
CARD.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "strict_candidate_retention_filter": ROOT / "runtime/model_selection/card/strict_candidate_retention_filter_v1/summary.json",
    "strict_retention_gate": ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json",
    "soxlbacktest_qlib_reference": ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/summary.json",
    "soxlbacktest_gate_summary": ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_gate_summary.csv",
    "three_fold_checkpoint_selector": ROOT / "runtime/model_selection/card/cap10_three_fold_checkpoint_drift_selector_v1/selection.json",
    "three_fold_checkpoint_by_fold": ROOT / "runtime/model_selection/card/cap10_three_fold_checkpoint_drift_selector_v1/checkpoint_selection_by_fold.csv",
    "joint_checkpoint_selector": ROOT / "runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/selection.json",
    "joint_checkpoint_by_fold": ROOT / "runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/checkpoint_selection_by_fold.csv",
    "joint_checkpoint_candidates": ROOT / "runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/checkpoint_candidate_metrics.csv",
    "state_multi_objective": ROOT / "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/summary.json",
    "state_multi_objective_candidates": ROOT / "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/candidate_summary.csv",
    "short_volshock_reward_checkpoint": ROOT / "runtime/reward/card/short_volshock_reward_checkpoint_screen_v1/summary.json",
    "short_volshock_drawdown_drift_recipe": ROOT / "runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1/selection.json",
    "joint_reward_checkpoint_recipe": ROOT / "runtime/model_selection/card/joint_reward_checkpoint_recipe_v1/summary.json",
    "warm_start_failure_handoff": ROOT / "runtime/model_selection/card/warm_start_official_failure_handoff_v1/summary.json",
    "prior_retraining_gate_gap_handoff": ROOT / "runtime/model_selection/card/retraining_gate_gap_handoff_v1/training_objective_handoff.json",
}

def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def summary_obj(path: Path) -> dict[str, Any]:
    payload = load_json(path)
    return payload.get("summary") if isinstance(payload.get("summary"), dict) else payload

def as_bool(value: Any) -> bool | None:
    if isinstance(value, bool):
        return value
    if value is None:
        return None
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return True
    if text in {"false", "0", "no"}:
        return False
    return None

source_manifest = pd.DataFrame(
    [
        {
            "source_label": label,
            "source_path": rel(path),
            "exists": path.exists(),
            "kind": "csv" if path.suffix == ".csv" else "json",
        }
        for label, path in SOURCES.items()
    ]
)
source_manifest.to_csv(CARD / "source_manifest.csv", index=False)
source_manifest


,source_label,source_path,exists,kind
0,strict_candidate_retention_filter,runtime/model_selection/card/strict_candidate_...,True,json
1,strict_retention_gate,runtime/model_selection/card/model_retention_w...,True,json
2,soxlbacktest_qlib_reference,runtime/model_selection/card/soxlbacktest_qlib...,True,json
3,soxlbacktest_gate_summary,runtime/model_selection/card/soxlbacktest_qlib...,True,csv
4,three_fold_checkpoint_selector,runtime/model_selection/card/cap10_three_fold_...,True,json
5,three_fold_checkpoint_by_fold,runtime/model_selection/card/cap10_three_fold_...,True,csv
6,joint_checkpoint_selector,runtime/model_selection/card/cap10_joint_contr...,True,json
7,joint_checkpoint_by_fold,runtime/model_selection/card/cap10_joint_contr...,True,csv
8,joint_checkpoint_candidates,runtime/model_selection/card/cap10_joint_contr...,True,csv
9,state_multi_objective,runtime/model_selection/card/state_conditioned...,True,json


In [2]:

strict = summary_obj(SOURCES["strict_candidate_retention_filter"])
retention_gate = summary_obj(SOURCES["strict_retention_gate"])
soxl_ref = summary_obj(SOURCES["soxlbacktest_qlib_reference"])
three_sel = load_json(SOURCES["three_fold_checkpoint_selector"])
joint_sel = load_json(SOURCES["joint_checkpoint_selector"])
state_summary = summary_obj(SOURCES["state_multi_objective"])
reward_checkpoint = summary_obj(SOURCES["short_volshock_reward_checkpoint"])
reward_recipe = load_json(SOURCES["short_volshock_drawdown_drift_recipe"])
joint_reward = summary_obj(SOURCES["joint_reward_checkpoint_recipe"])
warm_start = summary_obj(SOURCES["warm_start_failure_handoff"])
prior_handoff = load_json(SOURCES["prior_retraining_gate_gap_handoff"])

soxl_gate = pd.read_csv(SOURCES["soxlbacktest_gate_summary"])
three_by_fold = pd.read_csv(SOURCES["three_fold_checkpoint_by_fold"])
joint_by_fold = pd.read_csv(SOURCES["joint_checkpoint_by_fold"])
joint_candidates = pd.read_csv(SOURCES["joint_checkpoint_candidates"])
state_candidates = pd.read_csv(SOURCES["state_multi_objective_candidates"])

def evidence_row(
    source_label: str,
    obj: dict[str, Any],
    *,
    primary_blocker: str,
    key_metric_1: str = "",
    key_metric_2: str = "",
    selected_candidate: str | None = None,
    source_path: Path | None = None,
) -> dict[str, Any]:
    return {
        "source_label": source_label,
        "decision": obj.get("decision", ""),
        "promotion_allowed": obj.get("promotion_allowed"),
        "actor_retention_allowed": obj.get("actor_retention_allowed"),
        "training_allowed": obj.get("training_allowed_from_this_card", obj.get("training_allowed_next")),
        "ppo_allowed_next": obj.get("ppo_allowed_next"),
        "selected_candidate": selected_candidate or obj.get("selected_candidate") or obj.get("selected_recipe") or "",
        "primary_blocker": primary_blocker,
        "next_mechanism": obj.get("next_mechanism", ""),
        "key_metric_1": key_metric_1,
        "key_metric_2": key_metric_2,
        "source_path": rel(source_path or SOURCES[source_label]),
    }

risk_floor = soxl_gate.loc[soxl_gate["candidate"].eq("wf_pass_ytd18_15_15_02")].iloc[0]
legacy_default = soxl_gate.loc[soxl_gate["candidate"].eq("legacy_default")].iloc[0]
state_best = state_candidates.loc[state_candidates["candidate"].eq(state_summary.get("selected_candidate"))].iloc[0]
joint_ppo_bad = joint_candidates.loc[(joint_candidates["fold"].eq("wf_2022")) & (joint_candidates["candidate"].eq("ppo_retained_iter_4"))].iloc[0]

evidence_matrix = pd.DataFrame(
    [
        evidence_row(
            "strict_candidate_retention_filter",
            strict,
            primary_blocker="0 promotable profiles and 0 actor-retention allowed; PPO next is false.",
            key_metric_1=f"required_best_return_pct={strict.get('required_best_return_pct')}",
            key_metric_2=f"max_wf_drawdown_pct={strict.get('max_walkforward_drawdown_pct')}",
        ),
        evidence_row(
            "strict_retention_gate",
            retention_gate,
            primary_blocker="No scanned profile passes all required walk-forward windows and return/drawdown gates.",
            key_metric_1=f"passing_profile_count={retention_gate.get('passing_profile_count')}",
            key_metric_2=f"required_best_return_pct={retention_gate.get('required_best_return_pct')}",
        ),
        evidence_row(
            "soxlbacktest_qlib_reference",
            soxl_ref,
            primary_blocker="Qlib-aligned SOXLBacktest is a buy/sell reference, not a promotable actor.",
            key_metric_1=f"legacy_ytd_return_pct={legacy_default['ytd_return_pct']:.2f}",
            key_metric_2=f"risk_floor_worst_wf_max_dd_pct={risk_floor['worst_wf_max_drawdown_pct']:.2f}",
        ),
        evidence_row(
            "three_fold_checkpoint_selector",
            three_sel,
            primary_blocker="Three-fold selector only partially passes; wf_2020 fails target/candidate gate.",
            key_metric_1=f"wf_2020_relative_return={three_by_fold.loc[three_by_fold['fold'].eq('wf_2020'), 'selected_relative_return'].iloc[0]:.2f}",
            key_metric_2=f"wf_2020_max_dd={three_by_fold.loc[three_by_fold['fold'].eq('wf_2020'), 'selected_max_dd'].iloc[0]:.2f}",
        ),
        evidence_row(
            "joint_checkpoint_selector",
            joint_sel,
            primary_blocker="Joint selector is blocked by wf_2022 and PPO drift raises MaxDD materially.",
            key_metric_1=f"wf_2022_ppo_iter4_relative={joint_ppo_bad['test_relative_return']:.2f}",
            key_metric_2=f"wf_2022_ppo_iter4_max_dd={joint_ppo_bad['test_max_dd']:.2f}",
        ),
        evidence_row(
            "state_multi_objective",
            state_summary,
            primary_blocker="Best state-conditioned multi-objective candidate reaches relative target but fails drawdown gate.",
            key_metric_1=f"min_relative_return={state_best['min_relative_return']:.2f}",
            key_metric_2=f"max_max_dd={state_best['max_max_dd']:.2f}",
        ),
        evidence_row(
            "short_volshock_reward_checkpoint",
            reward_checkpoint,
            primary_blocker="Reward checkpoint screen remains no-promotion and no PPO next.",
            key_metric_1="selected=short_volshock_plain",
            key_metric_2="target_relative_return_pp=30,target_max_dd_pct=15",
        ),
        evidence_row(
            "short_volshock_drawdown_drift_recipe",
            reward_recipe,
            primary_blocker="Reward recipe is readiness for short screen only; it promotes no actor/checkpoint.",
            key_metric_1=f"drawdown_failure_capture={reward_recipe.get('drawdown_failure_capture'):.3f}",
            key_metric_2=f"trend_underparticipation_capture={reward_recipe.get('trend_underparticipation_capture'):.3f}",
            selected_candidate=reward_recipe.get("selected_recipe"),
            source_path=SOURCES["short_volshock_drawdown_drift_recipe"],
        ),
        evidence_row(
            "joint_reward_checkpoint_recipe",
            joint_reward,
            primary_blocker="Checkpoint objective is a scoring recipe only; no promotion evidence.",
            key_metric_1=f"selected_candidate={joint_reward.get('selected_candidate')}",
        ),
        evidence_row(
            "warm_start_failure_handoff",
            warm_start,
            primary_blocker="Official warm-start failure points to state-conditioned multi-objective training, not retention.",
            key_metric_1=f"next_mechanism={warm_start.get('next_mechanism')}",
        ),
        evidence_row(
            "prior_retraining_gate_gap_handoff",
            prior_handoff,
            primary_blocker="Prior handoff allowed training design only; promotion remains false.",
            key_metric_1=f"decision={prior_handoff.get('decision')}",
            key_metric_2=f"required_best_return_pct={prior_handoff.get('required_best_return_pct')}",
            source_path=SOURCES["prior_retraining_gate_gap_handoff"],
        ),
    ]
)

evidence_matrix.to_csv(CARD / "evidence_matrix.csv", index=False)
evidence_matrix


,source_label,decision,promotion_allowed,actor_retention_allowed,training_allowed,ppo_allowed_next,selected_candidate,primary_blocker,next_mechanism,key_metric_1,key_metric_2,source_path
0,strict_candidate_retention_filter,strict_candidate_retention_filter_no_promotabl...,None,None,None,False,,0 promotable profiles and 0 actor-retention al...,train_bull_trend_participation_with_drawdown_d...,required_best_return_pct=213.143462,max_wf_drawdown_pct=20.0,runtime/model_selection/card/strict_candidate_...
1,strict_retention_gate,,None,None,None,None,,No scanned profile passes all required walk-fo...,,passing_profile_count=0,required_best_return_pct=213.143462,runtime/model_selection/card/model_retention_w...
2,soxlbacktest_qlib_reference,soxlbacktest_reference_qlib_aligned_not_promot...,False,False,None,None,,Qlib-aligned SOXLBacktest is a buy/sell refere...,,legacy_ytd_return_pct=132.08,risk_floor_worst_wf_max_dd_pct=18.51,runtime/model_selection/card/soxlbacktest_qlib...
3,three_fold_checkpoint_selector,three_fold_checkpoint_selector_partial_pass_no...,False,False,False,None,,Three-fold selector only partially passes; wf_...,inspect_failed_fold_drawdown_or_reward_drift_b...,wf_2020_relative_return=-285.45,wf_2020_max_dd=19.42,runtime/model_selection/card/cap10_three_fold_...
4,joint_checkpoint_selector,checkpoint_drift_selector_partial_pass_wf2022_...,False,None,False,None,,Joint selector is blocked by wf_2022 and PPO d...,add_drawdown_drift_penalty_or_new_risk_state_b...,wf_2022_ppo_iter4_relative=41.77,wf_2022_ppo_iter4_max_dd=37.28,runtime/model_selection/card/cap10_joint_contr...
5,state_multi_objective,state_conditioned_multi_objective_drawdown_fai...,False,None,False,None,multi_obj_floor45_cap20_riskfirst,Best state-conditioned multi-objective candida...,revise risk-state representation or checkpoint...,min_relative_return=30.26,max_max_dd=30.63,runtime/model_selection/card/state_conditioned...
6,short_volshock_reward_checkpoint,reward_checkpoint_screen_no_promotion,False,None,False,False,short_volshock_plain,Reward checkpoint screen remains no-promotion ...,train_drawdown_drift_reward_or_checkpoint_recipe,selected=short_volshock_plain,"target_relative_return_pp=30,target_max_dd_pct=15",runtime/reward/card/short_volshock_reward_chec...
7,short_volshock_drawdown_drift_recipe,drawdown_drift_reward_recipe_ready_for_short_t...,False,None,True,None,dd1.5_thr15_pos0.05_tf1_floor0.25_lb63,Reward recipe is readiness for short screen on...,short_foldlocal_bc_or_ppo_training_screen,drawdown_failure_capture=0.874,trend_underparticipation_capture=1.000,runtime/reward/card/short_volshock_drawdown_dr...
8,joint_reward_checkpoint_recipe,checkpoint_objective_no_promotion,None,None,None,None,selected_router_no_equity_dd,Checkpoint objective is a scoring recipe only;...,,selected_candidate=selected_router_no_equity_dd,,runtime/model_selection/card/joint_reward_chec...
9,warm_start_failure_handoff,train_state_conditioned_multi_objective,False,None,False,None,,Official warm-start failure points to state-co...,train_state_conditioned_multi_objective,next_mechanism=train_state_conditioned_multi_o...,,runtime/model_selection/card/warm_start_offici...


In [3]:

fold_rows: list[dict[str, Any]] = []
for source_label, frame in [
    ("cap10_three_fold_checkpoint_drift_selector_v1", three_by_fold),
    ("cap10_joint_contract_checkpoint_drift_selector_v1", joint_by_fold),
]:
    for _, row in frame.iterrows():
        fold_rows.append(
            {
                "source_label": source_label,
                "fold": row.get("fold"),
                "selected_candidate": row.get("selected_candidate"),
                "relative_return_pp": row.get("selected_relative_return"),
                "max_dd_pct": row.get("selected_max_dd"),
                "target_gate": as_bool(row.get("selected_target_gate")),
                "drift_gate": as_bool(row.get("selected_drift_gate")),
                "candidate_gate": as_bool(row.get("selected_candidate_gate")),
                "blocker": "passes fold gates" if as_bool(row.get("selected_candidate_gate")) else "fold gate failed",
            }
        )

for _, row in soxl_gate.iterrows():
    fold_rows.append(
        {
            "source_label": "soxlbacktest_qlib_walkforward_baseline_v1",
            "fold": "wf_2020,wf_2022,wf_2023_2024",
            "selected_candidate": row.get("candidate"),
            "relative_return_pp": row.get("ytd_return_pct"),
            "max_dd_pct": row.get("worst_wf_max_drawdown_pct"),
            "target_gate": as_bool(row.get("ytd_current_best_alignment_pass")),
            "drift_gate": as_bool(row.get("wf_drawdown_pass")),
            "candidate_gate": as_bool(row.get("retention_gate_pass")),
            "blocker": row.get("failures", ""),
        }
    )

for _, row in state_candidates.iterrows():
    fold_rows.append(
        {
            "source_label": "state_conditioned_multi_objective_bc_screen_v1",
            "fold": "wf_2022,wf_2023_2024",
            "selected_candidate": row.get("candidate"),
            "relative_return_pp": row.get("min_relative_return"),
            "max_dd_pct": row.get("max_max_dd"),
            "target_gate": bool(row.get("min_relative_return", 0) >= 30.0),
            "drift_gate": bool(row.get("max_max_dd", 999) <= 20.0),
            "candidate_gate": bool(row.get("full_gate_all")),
            "blocker": "drawdown gate failed" if not bool(row.get("full_gate_all")) else "passes fold gates",
        }
    )

fold_gate_matrix = pd.DataFrame(fold_rows)
fold_gate_matrix.to_csv(CARD / "fold_gate_matrix.csv", index=False)
fold_gate_matrix


,source_label,fold,selected_candidate,relative_return_pp,max_dd_pct,target_gate,drift_gate,candidate_gate,blocker
0,cap10_three_fold_checkpoint_drift_selector_v1,wf_2020,bc_initialization,-285.449375,19.417943,False,True,False,fold gate failed
1,cap10_three_fold_checkpoint_drift_selector_v1,wf_2022,bc_initialization,47.131485,18.657201,True,True,True,passes fold gates
2,cap10_three_fold_checkpoint_drift_selector_v1,wf_2023_2024,bc_initialization,38.712278,10.403189,True,True,True,passes fold gates
3,cap10_joint_contract_checkpoint_drift_selector_v1,wf_2022,bc_initialization,36.419124,18.455088,False,True,False,fold gate failed
4,cap10_joint_contract_checkpoint_drift_selector_v1,wf_2023_2024,bc_initialization,30.501116,13.597475,True,True,True,passes fold gates
5,soxlbacktest_qlib_walkforward_baseline_v1,"wf_2020,wf_2022,wf_2023_2024",legacy_default,132.077977,84.918053,False,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
6,soxlbacktest_qlib_walkforward_baseline_v1,"wf_2020,wf_2022,wf_2023_2024",risk_cap_35_15_05,18.308910,43.271216,False,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_s...
7,soxlbacktest_qlib_walkforward_baseline_v1,"wf_2020,wf_2022,wf_2023_2024",low_risk_20_08_02,9.658820,24.732060,False,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_s...
8,soxlbacktest_qlib_walkforward_baseline_v1,"wf_2020,wf_2022,wf_2023_2024",wf_pass_ytd18_15_15_02,18.308910,18.514585,False,True,False,ytd_return_below_soxlbacktest_reference;ytd_re...
9,soxlbacktest_qlib_walkforward_baseline_v1,"wf_2020,wf_2022,wf_2023_2024",low_risk_15_05_02,6.032790,19.456050,False,True,False,ytd_return_below_soxlbacktest_reference;ytd_re...


In [4]:

stop_rules = pd.DataFrame(
    [
        {
            "rule": "strict_walkforward_gate",
            "trigger": "any required WF window missing or worst WF MaxDD >= 20%",
            "action": "no promotion and no actor retention",
            "evidence": "strict_candidate_retention_filter_v1 and model_retention_walkforward_gate_v1",
        },
        {
            "rule": "return_alignment_gate",
            "trigger": "YTD/common-window return below required_best_return_pct=213.143462 or below +30pp fold target",
            "action": "training result can remain diagnostic only",
            "evidence": "model_retention_walkforward_gate_v1; checkpoint selectors",
        },
        {
            "rule": "soxlbacktest_reference_boundary",
            "trigger": "candidate is SOXLBacktest/Qlib buy-sell reference",
            "action": "show as reference baseline only; never retain actor",
            "evidence": "soxlbacktest_qlib_walkforward_baseline_v1",
        },
        {
            "rule": "checkpoint_drift_stop",
            "trigger": "PPO checkpoint improves relative return but raises MaxDD drift materially",
            "action": "stop longer PPO; revise drawdown-drift reward/checkpoint objective first",
            "evidence": "cap10_joint_contract_checkpoint_drift_selector_v1 wf_2022 PPO iter 4",
        },
        {
            "rule": "recipe_not_actor",
            "trigger": "screen only produces reward/checkpoint recipe without official trained backtest",
            "action": "do not promote; allow only short fold-local training screen",
            "evidence": "short_volshock_drawdown_drift_reward_recipe_screen_v1",
        },
    ]
)
stop_rules.to_csv(CARD / "stop_rules.csv", index=False)

training_objective_recipe = {
    "experiment": "final_walkforward_retraining_handoff_v1",
    "decision": "short_foldlocal_drawdown_drift_training_screen_only_no_promotion",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "long_ppo_allowed": False,
    "short_training_screen_allowed": True,
    "reference_baseline": "soxl_backtest_policy_reimplementation_v1",
    "reference_role": "qlib_aligned_buy_sell_reference_only",
    "required_gates_for_future_retention": {
        "required_wf_windows": retention_gate.get("required_wf_windows", ["wf_2020", "wf_2022", "wf_2023_2024"]),
        "max_walkforward_drawdown_pct": float(strict.get("max_walkforward_drawdown_pct", 20.0)),
        "required_best_return_pct": float(strict.get("required_best_return_pct", 213.143462)),
        "target_relative_return_pp": 30.0,
        "internal_target_max_dd_pct": 15.0,
        "strict_candidate_actor_retention_allowed_count_min": 1,
    },
    "allowed_next_training_screen": {
        "scope": "short fold-local BC/PPO or checkpoint-scoring screen only",
        "changed_layer": ["reward", "checkpoint_selection", "risk_state_representation"],
        "objective_terms": [
            "drawdown_drift_penalty",
            "short_volshock_state_deleverage_consistency",
            "trend_preservation_floor",
            "checkpoint_objective_loss",
            "risk_state_representation_revision",
            "official_warm_start_walkforward_gate",
        ],
        "reward_objective_patch": reward_recipe.get("reward_objective_patch", {}),
        "source_recipe": "short_volshock_drawdown_drift_reward_recipe_screen_v1",
    },
    "blocked_paths": [
        "retain SOXLBacktest reference as actor",
        "promote any current replay/profile from strict_candidate_retention_filter_v1",
        "run longer PPO from cap10_joint_contract_checkpoint_drift_selector_v1 without drawdown-drift objective revision",
        "treat state_conditioned_multi_objective_bc_screen_v1 selected candidate as retained despite >20% MaxDD",
    ],
}

selection = {
    "experiment": "final_walkforward_retraining_handoff_v1",
    "decision": training_objective_recipe["decision"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_next": True,
    "next_mechanism": "short_foldlocal_bc_or_ppo_training_screen_with_drawdown_drift_reward_and_official_wf_gate",
    "selected_reference_baseline": "soxl_backtest_policy_reimplementation_v1",
    "selected_reward_recipe": reward_recipe.get("selected_recipe"),
    "primary_blocker": "strict retention has 0 promotable profiles and 0 actor-retention allowed; current screens are diagnostic only.",
    "sidecars": [
        "source_manifest.csv",
        "evidence_matrix.csv",
        "fold_gate_matrix.csv",
        "stop_rules.csv",
        "training_objective_recipe.json",
        "selection.json",
        "summary.json",
    ],
}

summary = {
    "experiment": "final_walkforward_retraining_handoff_v1",
    "decision": selection["decision"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "strict_candidate_retention": {
        "passing_profile_count": strict.get("passing_profile_count"),
        "promotable_profile_count": strict.get("promotable_profile_count"),
        "actor_retention_allowed_count": strict.get("actor_retention_allowed_count"),
        "ppo_allowed_next": strict.get("ppo_allowed_next"),
    },
    "soxlbacktest_reference": {
        "decision": soxl_ref.get("decision"),
        "promotion_allowed": soxl_ref.get("promotion_allowed"),
        "actor_retention_allowed": soxl_ref.get("actor_retention_allowed"),
    },
    "next_action": selection["next_mechanism"],
    "risk_notes": [
        "No current actor is final-retained under strict walk-forward retention.",
        "SOXLBacktest/Qlib evidence is useful as buy/sell baseline but not deployable actor evidence.",
        "The next training screen must optimize drawdown drift jointly with trend participation and remain no-promotion until official WF gates pass.",
    ],
}

for name, payload in [
    ("training_objective_recipe.json", training_objective_recipe),
    ("selection.json", selection),
    ("summary.json", summary),
]:
    (CARD / name).write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

stop_rules


,rule,trigger,action,evidence
0,strict_walkforward_gate,any required WF window missing or worst WF Max...,no promotion and no actor retention,strict_candidate_retention_filter_v1 and model...
1,return_alignment_gate,YTD/common-window return below required_best_r...,training result can remain diagnostic only,model_retention_walkforward_gate_v1; checkpoin...
2,soxlbacktest_reference_boundary,candidate is SOXLBacktest/Qlib buy-sell reference,show as reference baseline only; never retain ...,soxlbacktest_qlib_walkforward_baseline_v1
3,checkpoint_drift_stop,PPO checkpoint improves relative return but ra...,stop longer PPO; revise drawdown-drift reward/...,cap10_joint_contract_checkpoint_drift_selector...
4,recipe_not_actor,screen only produces reward/checkpoint recipe ...,do not promote; allow only short fold-local tr...,short_volshock_drawdown_drift_reward_recipe_sc...


In [5]:

conclusion = f"""
## Conclusion

Decision: `{selection['decision']}`.

Current retention state is still no-promotion: strict candidate retention reports
`passing_profile_count={strict.get('passing_profile_count')}`,
`promotable_profile_count={strict.get('promotable_profile_count')}`, and
`actor_retention_allowed_count={strict.get('actor_retention_allowed_count')}`.
SOXLBacktest/Qlib remains a reference buy/sell baseline only, not a retained actor.

The next allowed training work is a short fold-local BC/PPO or checkpoint-scoring
screen using the drawdown-drift reward recipe and trend-preservation floor. It
must validate with official warm-start walk-forward gates before any actor or
checkpoint is retained. Longer PPO from the current checkpoint selectors is
blocked because the wf_2022 PPO drift candidate raises MaxDD to
`{joint_ppo_bad['test_max_dd']:.2f}%` despite improving relative return.

Primary sidecars written:

- `source_manifest.csv`
- `evidence_matrix.csv`
- `fold_gate_matrix.csv`
- `stop_rules.csv`
- `training_objective_recipe.json`
- `selection.json`
- `summary.json`

Next action: implement or run only the constrained short fold-local drawdown-drift
training screen, then rerun strict candidate retention. Do not promote anything
from this notebook.
"""
display(Markdown(conclusion))



## Conclusion

Decision: `short_foldlocal_drawdown_drift_training_screen_only_no_promotion`.

Current retention state is still no-promotion: strict candidate retention reports
`passing_profile_count=0`,
`promotable_profile_count=0`, and
`actor_retention_allowed_count=0`.
SOXLBacktest/Qlib remains a reference buy/sell baseline only, not a retained actor.

The next allowed training work is a short fold-local BC/PPO or checkpoint-scoring
screen using the drawdown-drift reward recipe and trend-preservation floor. It
must validate with official warm-start walk-forward gates before any actor or
checkpoint is retained. Longer PPO from the current checkpoint selectors is
blocked because the wf_2022 PPO drift candidate raises MaxDD to
`37.28%` despite improving relative return.

Primary sidecars written:

- `source_manifest.csv`
- `evidence_matrix.csv`
- `fold_gate_matrix.csv`
- `stop_rules.csv`
- `training_objective_recipe.json`
- `selection.json`
- `summary.json`

Next action: implement or run only the constrained short fold-local drawdown-drift
training screen, then rerun strict candidate retention. Do not promote anything
from this notebook.
